# Experiment 4: Adaptive Retrieval Router

A rule-based router that picks a retrieval strategy **per query** instead of using one
fixed strategy for everything. Self-contained: re-loads `docs.jsonl` and
`qa_final_180.jsonl` and rebuilds Dense/BM25/Hybrid/Rerank from scratch.

**What this notebook does, in order:**
1. Rebuild the three base retrieval methods (Dense, Hybrid α=0.75, Hybrid+Rerank) from Experiments 2–3.
2. Compute the **oracle upper bound** — best possible accuracy if the correct method were picked for every query in hindsight.
3. Define and validate a **rule-based router** (regex + query-length heuristics) against the dataset's real category labels (confusion matrix).
4. Compare Router vs. each fixed "always-X" strategy on accuracy **and** a relative compute-cost metric.
5. Test statistical significance (Router vs. best fixed baseline).
6. Run an ablation (which router signal — keyword-regex or length — matters more).
7. Analyze errors caused specifically by router misclassification.

Run every cell top to bottom. Upload `docs.jsonl` and `qa_final_180.jsonl` when prompted.

In [1]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.9 MB/s eta 0:00:00


## Step 1: Upload data

In [3]:
from google.colab import drive
import os, json
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

# Your existing Drive folder
DATA_DIR = '/content/drive/MyDrive/AI_Project/RAG Research/Exp2/data'

# Check files
print("Files:", os.listdir(DATA_DIR))

# Load documents and QA
docs = [
    json.loads(l)
    for l in open(os.path.join(DATA_DIR, 'docs.jsonl'), encoding='utf-8')
]

qa = [
    json.loads(l)
    for l in open(os.path.join(DATA_DIR, 'qa_final_180.jsonl'), encoding='utf-8')
]

texts = [d['text'] for d in docs]
ids = [d['id'] for d in docs]

print(f"{len(docs)} docs, {len(qa)} questions")

Mounted at /content/drive
Files: ['docs.jsonl', 'qa_final_180.jsonl']
60 docs, 180 questions


## Step 2: Rebuild Dense, BM25, Hybrid, Cross-Encoder Rerank (same as Experiments 2–3)

In [4]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

tokenized_corpus = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
doc_text_by_id = {d['id']: d['text'] for d in docs}

def _norm(x):
    rng = x.max() - x.min()
    return (x - x.min()) / (rng + 1e-9)

ALPHA = 0.75

def rank_dense(query, k):
    q_emb = model.encode([query])
    _, idx = index.search(q_emb, k)
    return [ids[i] for i in idx[0]]

def rank_hybrid(query, k, alpha=ALPHA):
    bm25_scores = np.array(bm25.get_scores(query.lower().split()))
    q_emb = model.encode([query])
    vec_scores = -np.linalg.norm(doc_embeddings - q_emb, axis=1)
    combined = alpha * _norm(vec_scores) + (1 - alpha) * _norm(bm25_scores)
    top = np.argsort(combined)[::-1][:k]
    return [ids[i] for i in top]

def rank_hybrid_rerank(query, k, shortlist_size=10):
    shortlist = rank_hybrid(query, shortlist_size)
    pairs = [(query, doc_text_by_id[d]) for d in shortlist]
    scores = reranker.predict(pairs)
    order = np.argsort(scores)[::-1]
    return [shortlist[i] for i in order][:k]

METHODS = {"dense": rank_dense, "hybrid": rank_hybrid, "hybrid_rerank": rank_hybrid_rerank}
# Relative compute cost per query, approximate: dense=1 embedding, hybrid=1 embedding+BM25 (~same),
# hybrid_rerank=1 embedding+BM25+10 cross-encoder passes (cross-encoder pass costed at ~2x a dense embedding call)
COST = {"dense": 1.0, "hybrid": 1.1, "hybrid_rerank": 1.1 + 10*2.0}
print("Relative cost per method:", COST)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Relative cost per method: {'dense': 1.0, 'hybrid': 1.1, 'hybrid_rerank': 21.1}


## Step 3: Get full ranked lists (k=5) from all three methods, for every query — needed for oracle + router

In [5]:
results_by_method = {m: [] for m in METHODS}
for item in qa:
    q = item['question']
    for name, fn in METHODS.items():
        results_by_method[name].append(fn(q, 5))
print("Done. Example (query 0):")
for m in METHODS:
    print(m, results_by_method[m][0])

Done. Example (query 0):
dense ['doc1', 'doc25', 'doc42', 'doc4', 'doc45']
hybrid ['doc1', 'doc25', 'doc42', 'doc45', 'doc16']
hybrid_rerank ['doc1', 'doc45', 'doc26', 'doc4', 'doc28']


## Step 4: Metrics helper + Oracle upper bound

Oracle = for each query, check whether **any** of the three methods got the correct
document in the top-1 / top-3 / top-5. This is the ceiling a perfect router could reach
— it is NOT itself a retrieval method (it cheats by knowing the answer), it's a
diagnostic upper bound.

In [6]:
def hit_at_k(ranked, target, k):
    return int(target in ranked[:k])

def rr(ranked, target):
    return 1.0/(ranked.index(target)+1) if target in ranked else 0.0

targets = [item['expected_doc_id'] for item in qa]
categories = [item['category'] for item in qa]
n = len(qa)

def summarize(hits1, hits3, hits5, rrs):
    return {"recall@1": sum(hits1)/n, "recall@3": sum(hits3)/n, "recall@5": sum(hits5)/n, "mrr": sum(rrs)/n}

# Per-method summaries (sanity check against Experiments 2-3)
method_summary = {}
for m in METHODS:
    h1=[hit_at_k(r,t,1) for r,t in zip(results_by_method[m],targets)]
    h3=[hit_at_k(r,t,3) for r,t in zip(results_by_method[m],targets)]
    h5=[hit_at_k(r,t,5) for r,t in zip(results_by_method[m],targets)]
    rrs=[rr(r,t) for r,t in zip(results_by_method[m],targets)]
    method_summary[m] = summarize(h1,h3,h5,rrs)
    method_summary[m]['_h1'] = h1  # keep per-query hits for later significance tests

print(json.dumps({k:{kk:vv for kk,vv in v.items() if kk!='_h1'} for k,v in method_summary.items()}, indent=2))

# Oracle: best of the three per query
oracle_h1,oracle_h3,oracle_h5,oracle_rr = [],[],[],[]
for i in range(n):
    t = targets[i]
    best_rank_list = max(METHODS, key=lambda m: rr(results_by_method[m][i], t))
    r = results_by_method[best_rank_list][i]
    oracle_h1.append(hit_at_k(r,t,1)); oracle_h3.append(hit_at_k(r,t,3))
    oracle_h5.append(hit_at_k(r,t,5)); oracle_rr.append(rr(r,t))
oracle_summary = summarize(oracle_h1,oracle_h3,oracle_h5,oracle_rr)
print("\nORACLE (upper bound):", oracle_summary)

{
  "dense": {
    "recall@1": 0.7222222222222222,
    "recall@3": 0.8555555555555555,
    "recall@5": 0.9277777777777778,
    "mrr": 0.8012037037037038
  },
  "hybrid": {
    "recall@1": 0.7611111111111111,
    "recall@3": 0.9111111111111111,
    "recall@5": 0.9333333333333333,
    "mrr": 0.8306481481481482
  },
  "hybrid_rerank": {
    "recall@1": 0.7166666666666667,
    "recall@3": 0.8722222222222222,
    "recall@5": 0.9222222222222223,
    "mrr": 0.8
  }
}

ORACLE (upper bound): {'recall@1': 0.8666666666666667, 'recall@3': 0.9388888888888889, 'recall@5': 0.9722222222222222, 'mrr': 0.9071296296296296}


## Step 5: The router — rule-based, and validated against the real category labels

Router logic:
- Contains an HTTP status code (4xx/5xx) or an underscore-joined directive-like token
  (`word_word`), AND is short (<12 tokens) → predict **keyword** → use **dense only**
- 14+ tokens → predict **troubleshooting** → use **hybrid only** (skip rerank — Experiment 3
  showed reranking hurts this category)
- Otherwise → predict **semantic** → use **hybrid + rerank**

This is deliberately simple (no ML training) — a transparent, auditable baseline router.

In [7]:
def predict_category(query):
    toks = query.lower().split()
    has_code = bool(re.search(r'\b[45]\d\d\b', query)) or bool(re.search(r'[a-z]+_[a-z]+', query.lower()))
    if has_code and len(toks) < 12:
        return 'keyword'
    if len(toks) >= 14:
        return 'troubleshooting'
    return 'semantic'

ROUTE_METHOD = {'keyword': 'dense', 'troubleshooting': 'hybrid', 'semantic': 'hybrid_rerank'}

predicted_categories = [predict_category(item['question']) for item in qa]

# Confusion matrix: predicted vs true category
from collections import Counter
cats = ['keyword','semantic','troubleshooting']
confusion = {p: Counter() for p in cats}
for pred, true in zip(predicted_categories, categories):
    confusion[pred][true] += 1

import pandas as pd
confusion_df = pd.DataFrame({p: [confusion[p].get(t,0) for t in cats] for p in cats}, index=cats).T
confusion_df.index.name = 'true \\ predicted'
print("Router classification accuracy:", sum(p==t for p,t in zip(predicted_categories,categories))/n)
confusion_df

Router classification accuracy: 0.48333333333333334


,keyword,semantic,troubleshooting
true \ predicted,,,
keyword,23,0,0
semantic,28,4,0
troubleshooting,9,56,60


## Step 6: Evaluate the Router's actual retrieval accuracy (using its chosen method per query) + cost

In [8]:
router_h1,router_h3,router_h5,router_rr,router_cost = [],[],[],[],[]
for i in range(n):
    t = targets[i]
    chosen_method = ROUTE_METHOD[predicted_categories[i]]
    r = results_by_method[chosen_method][i]
    router_h1.append(hit_at_k(r,t,1)); router_h3.append(hit_at_k(r,t,3))
    router_h5.append(hit_at_k(r,t,5)); router_rr.append(rr(r,t))
    router_cost.append(COST[chosen_method])
router_summary = summarize(router_h1,router_h3,router_h5,router_rr)
router_summary['avg_cost'] = sum(router_cost)/n
print("ROUTER:", router_summary)

comparison_rows = []
for m in METHODS:
    row = {k:v for k,v in method_summary[m].items() if k!='_h1'}
    row['method'] = f'Always {m}'
    row['avg_cost'] = COST[m]
    comparison_rows.append(row)
comparison_rows.append({**oracle_summary, 'method':'Oracle (upper bound)', 'avg_cost': None})
comparison_rows.append({**{k:v for k,v in router_summary.items() if k!='avg_cost'}, 'method':'Router (proposed)', 'avg_cost': router_summary['avg_cost']})

comparison_df = pd.DataFrame(comparison_rows)[['method','recall@1','recall@3','recall@5','mrr','avg_cost']]
comparison_df

ROUTER: {'recall@1': 0.7611111111111111, 'recall@3': 0.9111111111111111, 'recall@5': 0.9333333333333333, 'mrr': 0.8315740740740741, 'avg_cost': 4.642777777777778}


,method,recall@1,recall@3,recall@5,mrr,avg_cost
0,Always dense,0.722222,0.855556,0.927778,0.801204,1.000000
1,Always hybrid,0.761111,0.911111,0.933333,0.830648,1.100000
2,Always hybrid_rerank,0.716667,0.872222,0.922222,0.800000,21.100000
3,Oracle (upper bound),0.866667,0.938889,0.972222,0.907130,NaN
4,Router (proposed),0.761111,0.911111,0.933333,0.831574,4.642778


## Step 7: Statistical significance — Router vs. the best fixed baseline

Same paired bootstrap + McNemar approach as Experiments 2–3, on per-query Recall@1.

In [9]:
def paired_bootstrap(a, b, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    nq = len(a)
    obs = a.mean() - b.mean()
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, nq, nq)
        diffs[i] = a[idx].mean() - b[idx].mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"observed_diff": obs, "ci_95": (lo, hi), "p_one_sided": float((diffs <= 0).mean())}

def mcnemar(a, b):
    a_r_b_w = int(((a==1)&(b==0)).sum()); a_w_b_r = int(((a==0)&(b==1)).sum())
    nd = a_r_b_w + a_w_b_r
    if nd == 0: return {"a_right_b_wrong":0,"a_wrong_b_right":0,"chi2":0.0,"p_value":1.0}
    chi2 = (abs(a_r_b_w - a_w_b_r)-1)**2 / nd
    from scipy.stats import chi2 as chi2_dist
    return {"a_right_b_wrong":a_r_b_w,"a_wrong_b_right":a_w_b_r,"chi2":chi2,"p_value":1-chi2_dist.cdf(chi2,df=1)}

router_arr = np.array(router_h1)
# best fixed baseline = whichever Always-X has the highest recall@1
best_fixed_name = max(METHODS, key=lambda m: method_summary[m]['recall@1'])
best_fixed_arr = np.array(method_summary[best_fixed_name]['_h1'])
print(f"Comparing Router vs. best fixed baseline: Always {best_fixed_name}")
boot = paired_bootstrap(router_arr, best_fixed_arr)
mcn = mcnemar(router_arr, best_fixed_arr)
print("Bootstrap:", boot)
print("McNemar:  ", mcn)

Comparing Router vs. best fixed baseline: Always hybrid
Bootstrap: {'observed_diff': np.float64(0.0), 'ci_95': (np.float64(-0.016666666666666607), np.float64(0.016666666666666607)), 'p_one_sided': 0.6461}
McNemar:   {'a_right_b_wrong': 1, 'a_wrong_b_right': 1, 'chi2': 0.5, 'p_value': np.float64(0.47950012218695337)}


## Step 8: Ablation — which router signal actually matters?

Three variants: keyword-regex signal alone, length signal alone, and both combined
(the full router from Step 5). Everything not caught by a signal falls back to
`hybrid_rerank` (the semantic-category default).

In [10]:
def route_regex_only(query):
    toks = query.lower().split()
    has_code = bool(re.search(r'\b[45]\d\d\b', query)) or bool(re.search(r'[a-z]+_[a-z]+', query.lower()))
    return 'dense' if (has_code and len(toks) < 12) else 'hybrid_rerank'

def route_length_only(query):
    toks = query.lower().split()
    return 'hybrid' if len(toks) >= 14 else 'hybrid_rerank'

def route_combined(query):
    return ROUTE_METHOD[predict_category(query)]

def eval_router_variant(route_fn):
    h1=[]
    for i,item in enumerate(qa):
        m = route_fn(item['question'])
        h1.append(hit_at_k(results_by_method[m][i], targets[i], 1))
    return sum(h1)/n

ablation = {
    "regex_signal_only": eval_router_variant(route_regex_only),
    "length_signal_only": eval_router_variant(route_length_only),
    "both_combined (proposed)": eval_router_variant(route_combined),
}
print(json.dumps(ablation, indent=2))

{
  "regex_signal_only": 0.7166666666666667,
  "length_signal_only": 0.7611111111111111,
  "both_combined (proposed)": 0.7611111111111111
}


## Step 9: Error analysis — accuracy cost of router misclassification

In [11]:
misclassified = [(i, predicted_categories[i], categories[i]) for i in range(n) if predicted_categories[i]!=categories[i]]
print(f"Router misclassified {len(misclassified)} / {n} queries ({len(misclassified)/n*100:.1f}%)")

mis_h1 = [router_h1[i] for i,_,_ in misclassified]
correct_class_h1 = [router_h1[i] for i in range(n) if i not in [m[0] for m in misclassified]]
print(f"Recall@1 on misclassified queries: {sum(mis_h1)/max(len(mis_h1),1):.3f} (n={len(mis_h1)})")
print(f"Recall@1 on correctly-classified queries: {sum(correct_class_h1)/max(len(correct_class_h1),1):.3f} (n={len(correct_class_h1)})")

error_rows = [{"question": qa[i]['question'], "predicted_category": p, "true_category": t,
               "method_used": ROUTE_METHOD[p], "recall@1_hit": router_h1[i]}
              for i,p,t in misclassified]
error_df = pd.DataFrame(error_rows)
error_df

Router misclassified 93 / 180 queries (51.7%)
Recall@1 on misclassified queries: 0.742 (n=93)
Recall@1 on correctly-classified queries: 0.782 (n=87)


,question,predicted_category,true_category,method_used,recall@1_hit
0,Nginx keeps hitting the worker_connections lim...,troubleshooting,keyword,hybrid,1
1,Bahut saare users ek saath aane par server nay...,troubleshooting,semantic,hybrid,0
2,"Getting intermittent 502 Bad Gateway errors, w...",semantic,keyword,hybrid_rerank,1
3,Site kabhi kabhi ek generic gateway error dikh...,troubleshooting,semantic,hybrid,1
4,Ek report jo generate hone mein bahut time let...,troubleshooting,semantic,hybrid,0
...,...,...,...,...,...
88,Ek script/module ko ek API key chahiye jo envi...,troubleshooting,semantic,hybrid,1
89,How do I make one server block respond to mult...,troubleshooting,keyword,hybrid,1
90,Kai alag alag domains hain jinko exactly same ...,troubleshooting,semantic,hybrid,1
91,How does setting error_log to 'debug' level he...,troubleshooting,keyword,hybrid,1


## Step 10: Save everything

In [12]:
exp4_export = {
    "method_summary": {m:{k:v for k,v in s.items() if k!='_h1'} for m,s in method_summary.items()},
    "oracle_summary": oracle_summary,
    "router_summary": router_summary,
    "router_classification_accuracy": sum(p==t for p,t in zip(predicted_categories,categories))/n,
    "significance_router_vs_best_fixed": {"baseline": best_fixed_name,
        "bootstrap": {"observed_diff": boot['observed_diff'], "ci_95": list(boot['ci_95']), "p_one_sided": boot['p_one_sided']},
        "mcnemar": mcn},
    "ablation": ablation,
    "misclassification_count": len(misclassified),
    "misclassification_rate": len(misclassified)/n,
}
with open('/content/experiment4_results.json','w') as f:
    json.dump(exp4_export, f, indent=2, default=float)

confusion_df.to_csv('/content/experiment4_confusion_matrix.csv')
comparison_df.to_csv('/content/experiment4_comparison_table.csv', index=False)
error_df.to_csv('/content/experiment4_router_errors.csv', index=False)

print("Saved: experiment4_results.json, experiment4_confusion_matrix.csv,")
print("       experiment4_comparison_table.csv, experiment4_router_errors.csv")
print("Download all four from the Colab Files panel and send them back.")

Saved: experiment4_results.json, experiment4_confusion_matrix.csv,
       experiment4_comparison_table.csv, experiment4_router_errors.csv
Download all four from the Colab Files panel and send them back.


## What to report in the paper
- **Comparison table** (Step 6) — Always-Dense / Always-Hybrid / Always-Hybrid+Rerank / Router / Oracle, accuracy *and* cost side by side. The Router-vs-Oracle gap shows how much headroom is left; the Router-vs-best-fixed comparison shows the actual proposed contribution.
- **Confusion matrix** (Step 5) — how accurate the router's category classification is on its own, independent of retrieval accuracy.
- **Significance test** (Step 7) — is the router's accuracy improvement (if any) over the best fixed baseline real at n=180, exactly as in Experiments 2–3.
- **Ablation** (Step 8) — which signal (keyword-regex vs. length) is doing the work.
- **Error analysis** (Step 9) — accuracy specifically on the queries the router misclassifies, which isolates classification error from retrieval error.
- If the router doesn't beat the best fixed strategy, or isn't significant, report that plainly — a well-instrumented negative result (with the oracle gap and ablation explaining why) is still a genuine contribution, and is exactly the kind of honest reporting the rest of this paper already does.